# Teen Rights and Law Lab - Colab Prototype

This notebook is a Colab-ready starter prototype for **Rights Guide**, an AI-based law education app for teenagers.

Rights Guide is designed to:

- explain laws and rights in plain teen-friendly language;
- answer general law and rights questions through an AI character;
- offer mini-games and quizzes for criminal law, family law, civil law, school rights, and digital rights;
- recommend learning paths based on interests and knowledge gaps;
- support community-style discussion with safety moderation;
- prepare the core Python logic for later packaging into a website app and iOS app.

**Important legal and safety note:** This prototype gives educational legal information only. It is not legal advice, not a lawyer, and not a substitute for a trusted adult, school counselor, legal aid organization, or attorney. Laws depend heavily on country, state, city, school policy, age, and specific facts.

## Project Credits

**Author:** Arya Patel  
**Mentor:** Dr. Qingyang Xiao

> The app and AI assistant are not branded with the author's first name. The name appears only in the project credit.


## Project Roadmap Toward August

Suggested milestone plan:

| Phase | Target | Output |
|---|---:|---|
| Concept freeze | Now | Product scope, safety rules, topic list, user flows |
| Core Python prototype | May-June | NLP router, Rights Guide agent, quiz engine, profile logic, moderation |
| AI improvement | June-July | Better response templates, safer knowledge base, testing set |
| App packaging | July-August | Website API, UI prototype, iOS wrapper plan |
| Open source setup | July-August | GitHub repo, README, LICENSE, contribution rules |
| IP/copyright preparation | By August | Rights Guide name/art/content ownership checklist; consult qualified counsel |

Open source and copyright are different issues. You can open source code while still owning copyright in original code, text, art, and design. You should choose a license intentionally and consult a qualified IP professional for brand, copyright, and possible trademark questions.

## 1. Colab Setup

Run this notebook top to bottom in Google Colab. The prototype avoids paid APIs and heavy LLM dependencies. It uses a simple, transparent, hard-coded NLP approach so the team can inspect and improve every part.

In [ ]:
# Colab setup cell.
# In most Colab runtimes, numpy, pandas, and scikit-learn are already installed.
# Uncomment the line below if a package is missing.
# !pip -q install numpy pandas scikit-learn

import re
import random
from dataclasses import dataclass, field
from typing import Dict, List, Any, Tuple, Optional

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

APP_NAME = "Teen Rights and Law Lab"
VERSION = "0.1.0"

print(f"{APP_NAME} prototype loaded. Version: {VERSION}")

## 2. Product Safety Principles

Because Rights Guide is for teenagers, safety is part of the core AI and cannot be added later.

Recommended principles:

1. **Educational only:** Rights Guide explains general legal concepts, not personalized legal advice.
2. **Jurisdiction-aware:** Rights Guide should ask for country/state only when needed and avoid pretending one rule applies everywhere.
3. **Minor privacy:** Do not collect unnecessary personal data. Avoid precise location, full name, school ID, phone number, or private family details.
4. **No sensitive clustering:** Do not cluster teenagers by protected or sensitive demographics. Prefer clustering by interests, learning goals, and quiz knowledge gaps.
5. **Escalation:** Abuse, danger, self-harm, exploitation, emergency, or active crime situations should trigger trusted-adult / emergency / professional help guidance.
6. **Community safety:** Use moderation, reporting, anti-bullying rules, and human review before allowing public posts.
7. **No evasion help:** Rights Guide should not help users break laws, avoid consequences, threaten others, or hide wrongdoing.

In [ ]:
SAFETY_CONFIG = {
    "emergency_keywords": [
        "emergency", "danger", "hurt me", "hurting me", "abuse", "abused", "assault",
        "self harm", "suicide", "kill myself", "kill me", "threatened", "weapon",
        "stalking", "exploited", "blackmail", "trafficking", "kidnap", "run away"
    ],
    "illegal_help_keywords": [
        "how do i hide", "how to hide", "avoid getting caught", "fake evidence",
        "lie to police", "hack", "steal", "shoplift", "make a fake id", "buy drugs",
        "sell drugs", "beat someone", "threaten someone"
    ],
    "personal_legal_advice_keywords": [
        "should i sue", "will i win", "am i guilty", "can i get away", "what should i plead",
        "how much money will i get", "do i need a lawyer", "should i confess"
    ]
}

BASE_DISCLAIMER = (
    "I can share general legal education in plain language, but I am not a lawyer and this is not legal advice. "
    "Rules can change by place, age, school policy, and facts."
)

EMERGENCY_RESPONSE = (
    "I am sorry you are dealing with something serious. If you are in immediate danger, contact local emergency services now. "
    "If you are a teen, try to reach a trusted adult, school counselor, guardian, legal aid group, or child protection hotline in your area. "
    "Rights Guide can explain general rights, but urgent safety needs should go to real people who can help right away."
)

ILLEGAL_HELP_RESPONSE = (
    "I cannot help with breaking the law, hiding wrongdoing, threatening people, or avoiding safety rules. "
    "I can help explain the law, safer choices, consequences, and how to ask for help responsibly."
)

def contains_any(text: str, keywords: List[str]) -> bool:
    t = text.lower()
    return any(k in t for k in keywords)

def safety_guardrail(user_text: str) -> Optional[str]:
    """Return a safety response if the user text triggers a high-priority safety rule."""
    if contains_any(user_text, SAFETY_CONFIG["emergency_keywords"]):
        return EMERGENCY_RESPONSE
    if contains_any(user_text, SAFETY_CONFIG["illegal_help_keywords"]):
        return ILLEGAL_HELP_RESPONSE
    return None

def legal_advice_guardrail(user_text: str) -> bool:
    return contains_any(user_text, SAFETY_CONFIG["personal_legal_advice_keywords"])

# Quick smoke tests
for test in ["Can you explain my rights?", "How do I make a fake ID?", "I am in danger and someone hurt me"]:
    print("Input:", test)
    print("Guardrail:", safety_guardrail(test))
    print()

## 3. Hard-Coded Legal Education Knowledge Base

This is a starter knowledge base. The team should have qualified reviewers check every legal statement before release. For launch, consider writing content by jurisdiction and showing a clear country/state selector.

The content below is deliberately general and simplified.

In [ ]:
KNOWLEDGE_BASE: Dict[str, Dict[str, Any]] = {
    "criminal_law": {
        "label": "Criminal Law",
        "teen_summary": "Criminal law is about actions the government treats as crimes, like theft, assault, vandalism, or drug offenses.",
        "plain_rights": [
            "You usually have the right to stay silent when questioned by police.",
            "You can ask for a parent, guardian, trusted adult, or lawyer depending on the situation and local law.",
            "You should not sign something you do not understand.",
            "Laws and police rules can be different for minors and adults."
        ],
        "common_questions": [
            "What happens if a teen gets arrested?",
            "Can police question me without my parents?",
            "What does it mean to stay silent?",
            "What is vandalism?",
            "What is shoplifting?"
        ],
        "teen_example": "If someone says, 'Just tell the police everything and it will be fine,' Rights Guide should explain that staying calm, asking for help, and understanding your rights matters.",
        "keywords": "police arrest crime criminal court juvenile offense shoplifting theft vandalism assault search silent lawyer"
    },
    "family_law": {
        "label": "Family Law",
        "teen_summary": "Family law deals with parents, guardians, custody, support, adoption, safety at home, and sometimes what happens when adults separate.",
        "plain_rights": [
            "Teens deserve safety at home and should ask for help if someone is hurting them.",
            "Custody and guardianship decisions are made by courts based on legal rules and the child's best interests.",
            "A teen's opinion may matter in some family situations, but the rules depend on local law."
        ],
        "common_questions": [
            "Can I choose which parent to live with?",
            "What is custody?",
            "What should I do if home feels unsafe?",
            "What is guardianship?",
            "Can a teen be adopted?"
        ],
        "teen_example": "If a teen says home feels unsafe, Rights Guide should prioritize safety and trusted adult support, not just legal definitions.",
        "keywords": "family custody parent guardian adoption home unsafe divorce support guardianship child protection"
    },
    "civil_law": {
        "label": "Civil Law",
        "teen_summary": "Civil law is about disputes between people or organizations, often involving money, contracts, property, or harm.",
        "plain_rights": [
            "A civil case is different from a criminal case; it often asks who owes money or who is responsible for harm.",
            "Contracts can matter, even online, but minors may have special rules.",
            "Keeping records like messages, receipts, and dates can help a trusted adult or lawyer understand what happened."
        ],
        "common_questions": [
            "What is a contract?",
            "Can a teen sign a contract?",
            "What is small claims court?",
            "What happens if someone damages my property?",
            "What is a lawsuit?"
        ],
        "teen_example": "If a friend breaks your phone and refuses to pay, Rights Guide can explain civil disputes, evidence, and asking an adult for help.",
        "keywords": "civil lawsuit contract property damage money dispute small claims sue negligence responsibility"
    },
    "school_rights": {
        "label": "School Rights",
        "teen_summary": "School rights involve discipline, searches, bullying, free expression, disability support, and fair treatment in school settings.",
        "plain_rights": [
            "Students have rights, but schools also have rules to keep students safe.",
            "Search rules at school can be different from searches outside school.",
            "Bullying, harassment, and discrimination should be reported to trusted adults or school officials.",
            "Students with disabilities may have rights to support and accommodations."
        ],
        "common_questions": [
            "Can school search my backpack?",
            "What can I do about bullying?",
            "Can I be suspended?",
            "What is an accommodation?",
            "Can students protest at school?"
        ],
        "teen_example": "If a teacher takes your phone, Rights Guide should explain that school policy, safety, and local law all matter.",
        "keywords": "school student backpack search bullying suspension expulsion accommodation disability teacher principal phone protest"
    },
    "digital_rights": {
        "label": "Digital Rights",
        "teen_summary": "Digital rights are about privacy, online speech, cyberbullying, sharing photos, accounts, platforms, and digital safety.",
        "plain_rights": [
            "Be careful before sharing private images, addresses, passwords, or school details.",
            "Cyberbullying and harassment can have school and legal consequences.",
            "Screenshots, messages, timestamps, and usernames can be important records.",
            "Platform rules are not the same as laws, but both can affect what happens."
        ],
        "common_questions": [
            "What is cyberbullying?",
            "Can someone post my photo without asking?",
            "What should I do if someone is blackmailing me online?",
            "Can school punish me for online posts?",
            "How do I protect my privacy?"
        ],
        "teen_example": "If someone threatens to leak private photos, Rights Guide should trigger a safety response and suggest trusted adult or professional support.",
        "keywords": "online internet cyberbullying privacy photo account password social media post harassment blackmail screenshot digital"
    }
}

for key, item in KNOWLEDGE_BASE.items():
    print(f"{key}: {item['label']} - {item['teen_summary']}")

## 4. NLP Topic Router

This first prototype uses TF-IDF similarity to route a teen's question to the most relevant law topic. It is transparent and easy to debug.

Later, the team can replace or supplement this with a fine-tuned NLP model or an approved LLM/RAG system.

In [ ]:
class TopicRouter:
    """Simple transparent NLP topic router using TF-IDF similarity."""

    def __init__(self, knowledge_base: Dict[str, Dict[str, Any]]):
        self.knowledge_base = knowledge_base
        self.topic_keys = list(knowledge_base.keys())
        self.documents = []
        for key in self.topic_keys:
            item = knowledge_base[key]
            text = " ".join([
                item["label"],
                item["teen_summary"],
                " ".join(item["plain_rights"]),
                " ".join(item["common_questions"]),
                item["teen_example"],
                item["keywords"]
            ])
            self.documents.append(text)
        self.vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
        self.matrix = self.vectorizer.fit_transform(self.documents)

    def predict(self, user_text: str, top_k: int = 3) -> List[Tuple[str, float]]:
        q_vec = self.vectorizer.transform([user_text])
        sims = cosine_similarity(q_vec, self.matrix).flatten()
        ranked_idx = np.argsort(sims)[::-1][:top_k]
        return [(self.topic_keys[i], float(sims[i])) for i in ranked_idx]

router = TopicRouter(KNOWLEDGE_BASE)

sample_questions = [
    "Can school search my backpack?",
    "What happens if police question me?",
    "Someone posted my picture online without permission.",
    "Can I choose which parent to live with?",
    "My friend broke my phone and will not pay me back."
]

for q in sample_questions:
    print("Question:", q)
    print("Top topics:", router.predict(q))
    print()

## 5. Rights Guide AI Character Agent

The agent below is not a full LLM. It is a transparent hard-coded core that combines:

- safety guardrails;
- topic routing;
- intent detection;
- teen-friendly response templates;
- quiz suggestions;
- escalation language for personal legal advice.

This is a safe base for the first product build. Later, a larger AI model can be added behind the same guardrails.

In [ ]:
def detect_intent(user_text: str) -> str:
    t = user_text.lower()
    if any(w in t for w in ["quiz", "game", "test me", "practice"]):
        return "quiz"
    if any(w in t for w in ["what is", "define", "meaning", "explain"]):
        return "explain"
    if any(w in t for w in ["can i", "can they", "can school", "can police", "can teacher", "can my", "am i allowed", "is it legal", "rights"]):
        return "rights"
    if any(w in t for w in ["happened", "my friend", "my parent", "teacher", "police", "someone"]):
        return "scenario"
    return "general"

@dataclass
class UserLearningState:
    nickname: str = "friend"
    age_band: str = "teen"
    preferred_tone: str = "friendly"  # friendly, direct, game-like
    interests: List[str] = field(default_factory=list)
    weak_topics: Dict[str, int] = field(default_factory=dict)
    completed_quizzes: int = 0
    positive_feedback: int = 0
    negative_feedback: int = 0

class RightsGuideAgent:
    def __init__(self, knowledge_base: Dict[str, Dict[str, Any]], router: TopicRouter):
        self.kb = knowledge_base
        self.router = router

    def answer(self, user_text: str, state: Optional[UserLearningState] = None) -> Dict[str, Any]:
        if state is None:
            state = UserLearningState()

        safety = safety_guardrail(user_text)
        if safety:
            return {
                "type": "safety",
                "topic": None,
                "response": safety,
                "confidence": 1.0,
                "next_steps": ["Reach a trusted adult or local emergency help if needed."]
            }

        intent = detect_intent(user_text)
        ranked = self.router.predict(user_text, top_k=3)
        topic_key, confidence = ranked[0]
        topic = self.kb[topic_key]

        advice_warning = ""
        if legal_advice_guardrail(user_text) or intent == "scenario":
            advice_warning = (
                "\n\nBecause this sounds personal, do not rely on an app alone. "
                "Talk with a trusted adult, school counselor, legal aid group, or lawyer who knows your local rules."
            )

        response = self._build_response(user_text, topic_key, intent, state)
        response = response + "\n\n" + BASE_DISCLAIMER + advice_warning

        return {
            "type": "answer",
            "topic": topic_key,
            "topic_label": topic["label"],
            "intent": intent,
            "confidence": round(float(confidence), 3),
            "response": response,
            "next_steps": self._suggest_next_steps(topic_key, intent)
        }

    def _build_response(self, user_text: str, topic_key: str, intent: str, state: UserLearningState) -> str:
        topic = self.kb[topic_key]
        intro = f"Hey {state.nickname}, Rights Guide here. This sounds related to {topic['label']}."

        if intent == "quiz":
            return intro + " I can turn this into a quick practice game. Try the quiz section below for this topic."

        if intent == "explain":
            rights_text = " ".join([f"- {r}" for r in topic["plain_rights"][:3]])
            return (
                intro + "\n\nPlain-language version: " + topic["teen_summary"] +
                "\n\nKey things to know:\n" + rights_text
            )

        if intent == "rights":
            rights = "\n".join([f"{i+1}. {r}" for i, r in enumerate(topic["plain_rights"])])
            return intro + " Here are general rights-related points:\n" + rights

        if intent == "scenario":
            return (
                intro + "\n\nA safe way to think about this is: first, make sure nobody is in immediate danger; "
                "second, write down what happened while it is fresh; third, ask a trusted adult or qualified professional. "
                "The law can depend on your location, age, school rules, and details.\n\n"
                f"Related idea: {topic['teen_summary']}"
            )

        return (
            intro + "\n\n" + topic["teen_summary"] +
            "\n\nTry asking: " + random.choice(topic["common_questions"])
        )

    def _suggest_next_steps(self, topic_key: str, intent: str) -> List[str]:
        topic = self.kb[topic_key]
        steps = [
            f"Play a 3-question mini-game on {topic['label']}.",
            "Save important words and definitions to your learning board.",
            "Ask a trusted adult or professional for real-life legal issues."
        ]
        if intent == "scenario":
            steps.insert(0, "Write down what happened, when, where, and who was involved.")
        return steps

rights_guide = RightsGuideAgent(KNOWLEDGE_BASE, router)
state = UserLearningState(nickname="Jordan", preferred_tone="friendly", interests=["school_rights", "digital_rights"])

for q in sample_questions:
    result = rights_guide.answer(q, state)
    print("USER:", q)
    print("TOPIC:", result.get("topic_label"), "CONF:", result["confidence"])
    print("RIGHTS GUIDE:", result["response"])
    print("NEXT:", result["next_steps"])
    print("-" * 80)

## 6. Mini-Game and Quiz Engine

This module turns law topics into practice questions. It tracks weak areas so Rights Guide can recommend more practice.

In [ ]:
QUIZ_BANK = {
    "criminal_law": [
        {
            "question": "What is the safest general response if police are questioning you and you feel confused?",
            "choices": [
                "Guess answers so the conversation ends fast",
                "Stay calm and ask for a trusted adult or lawyer if allowed",
                "Sign anything they give you",
                "Post about it online immediately"
            ],
            "answer_index": 1,
            "explanation": "Staying calm and asking for help is safer than guessing, signing, or posting details online."
        },
        {
            "question": "Criminal law usually involves:",
            "choices": ["Government accusations of crimes", "Only friendship arguments", "Only school homework", "Video game rules"],
            "answer_index": 0,
            "explanation": "Criminal law involves acts treated as crimes by the government."
        },
        {
            "question": "Which item is usually helpful to remember after an incident?",
            "choices": ["Dates and times", "Rumors only", "Deleted messages only", "A fake name"],
            "answer_index": 0,
            "explanation": "Accurate records like dates, times, messages, and names can help a trusted adult or lawyer understand what happened."
        }
    ],
    "family_law": [
        {
            "question": "Family law can include:",
            "choices": ["Custody and guardianship", "Only traffic tickets", "Only video games", "Only sports rules"],
            "answer_index": 0,
            "explanation": "Family law often covers custody, guardianship, support, adoption, and safety at home."
        },
        {
            "question": "If home feels unsafe, a teen should usually:",
            "choices": ["Ignore it forever", "Talk to a trusted adult or emergency resource", "Post private details publicly", "Run into danger"],
            "answer_index": 1,
            "explanation": "Safety comes first. A trusted adult, counselor, or emergency resource may help."
        },
        {
            "question": "Custody decisions are usually made by:",
            "choices": ["Courts using legal rules", "Random online polls", "Only the youngest sibling", "A social media platform"],
            "answer_index": 0,
            "explanation": "Courts use legal rules and facts to make custody decisions."
        }
    ],
    "civil_law": [
        {
            "question": "Civil law often focuses on:",
            "choices": ["Money, harm, contracts, or responsibility", "Only school lunch", "Only movie ratings", "Only sports scores"],
            "answer_index": 0,
            "explanation": "Civil law often deals with private disputes and responsibility for harm."
        },
        {
            "question": "If someone damages your property, what is a smart first step?",
            "choices": ["Destroy their property back", "Keep records and ask a trusted adult for help", "Make fake evidence", "Threaten them online"],
            "answer_index": 1,
            "explanation": "Records and adult help are safer than retaliation or threats."
        },
        {
            "question": "A contract is best described as:",
            "choices": ["An agreement with legal importance", "A meme", "A random rumor", "A video filter"],
            "answer_index": 0,
            "explanation": "A contract is an agreement that can have legal effects."
        }
    ],
    "school_rights": [
        {
            "question": "School search rules can be:",
            "choices": ["Different from rules outside school", "Exactly the same everywhere", "Chosen by students only", "Never allowed under any facts"],
            "answer_index": 0,
            "explanation": "Schools often have special rules because they must keep students safe."
        },
        {
            "question": "A good response to bullying is:",
            "choices": ["Save evidence and tell a trusted adult or school official", "Bully back harder", "Share private info online", "Do nothing forever"],
            "answer_index": 0,
            "explanation": "Saving evidence and reporting safely is usually better than escalating."
        },
        {
            "question": "An accommodation is often related to:",
            "choices": ["Support for a disability or learning need", "A new phone case", "A sports score", "A fashion rule"],
            "answer_index": 0,
            "explanation": "Accommodations can help students access school fairly."
        }
    ],
    "digital_rights": [
        {
            "question": "If someone threatens to share private images, what should come first?",
            "choices": ["Safety and trusted adult/professional help", "Sending more images", "Deleting every record", "Threatening them back"],
            "answer_index": 0,
            "explanation": "Safety comes first. Saving evidence and getting trusted help is important."
        },
        {
            "question": "Cyberbullying can lead to:",
            "choices": ["School and legal consequences", "No consequences ever", "Free concert tickets", "Better grades automatically"],
            "answer_index": 0,
            "explanation": "Cyberbullying and harassment can have serious consequences."
        },
        {
            "question": "A good privacy habit is:",
            "choices": ["Protect passwords and avoid sharing private details", "Post your address publicly", "Use the same password everywhere", "Give strangers your school schedule"],
            "answer_index": 0,
            "explanation": "Strong privacy habits reduce risk online."
        }
    ]
}

def get_quiz(topic_key: str, n: int = 3) -> List[Dict[str, Any]]:
    questions = QUIZ_BANK.get(topic_key, [])
    if not questions:
        return []
    return random.sample(questions, k=min(n, len(questions)))

def grade_quiz(topic_key: str, selected_indices: List[int], state: UserLearningState, quiz: Optional[List[Dict[str, Any]]] = None) -> Dict[str, Any]:
    if quiz is None:
        quiz = get_quiz(topic_key, n=len(selected_indices))
    correct = 0
    review = []
    for q, selected in zip(quiz, selected_indices):
        is_correct = selected == q["answer_index"]
        correct += int(is_correct)
        review.append({
            "question": q["question"],
            "selected": q["choices"][selected] if 0 <= selected < len(q["choices"]) else "Invalid choice",
            "correct": q["choices"][q["answer_index"]],
            "is_correct": is_correct,
            "explanation": q["explanation"]
        })
    score = correct / max(1, len(quiz))
    state.completed_quizzes += 1
    if score < 0.8:
        state.weak_topics[topic_key] = state.weak_topics.get(topic_key, 0) + 1
    return {
        "topic": topic_key,
        "score": score,
        "correct": correct,
        "total": len(quiz),
        "review": review,
        "weak_topics": state.weak_topics
    }

# Demo quiz
quiz_topic = "digital_rights"
quiz = get_quiz(quiz_topic)
for i, q in enumerate(quiz, start=1):
    print(f"Q{i}: {q['question']}")
    for j, c in enumerate(q["choices"]):
        print(f"  {j}. {c}")
    print()

# Simulate answer choices
print(grade_quiz(quiz_topic, [0, 0, 1], state, quiz=quiz))

## 7. Interest and Learning-Style Clustering

The original product idea included clustering teens by demographics, interests, understanding, and study habits. Because the users are minors, this prototype avoids demographic clustering. It clusters only by non-sensitive learning preferences, selected interests, and quiz knowledge gaps.

Production rule: do not collect or infer sensitive demographic attributes unless there is a clear legal, ethical, and product reason, and even then use strict consent and privacy controls.

In [ ]:
# Synthetic, non-sensitive sample profiles for clustering.
# Do not use real teen personal data in prototype notebooks.
profiles = pd.DataFrame([
    {"user_id": "u001", "likes_games": 1, "likes_voice": 1, "criminal_law": 2, "family_law": 0, "civil_law": 1, "school_rights": 4, "digital_rights": 5},
    {"user_id": "u002", "likes_games": 1, "likes_voice": 0, "criminal_law": 5, "family_law": 1, "civil_law": 2, "school_rights": 2, "digital_rights": 1},
    {"user_id": "u003", "likes_games": 0, "likes_voice": 1, "criminal_law": 0, "family_law": 5, "civil_law": 1, "school_rights": 2, "digital_rights": 0},
    {"user_id": "u004", "likes_games": 1, "likes_voice": 1, "criminal_law": 1, "family_law": 1, "civil_law": 0, "school_rights": 5, "digital_rights": 4},
    {"user_id": "u005", "likes_games": 0, "likes_voice": 0, "criminal_law": 1, "family_law": 2, "civil_law": 5, "school_rights": 0, "digital_rights": 1},
    {"user_id": "u006", "likes_games": 1, "likes_voice": 0, "criminal_law": 4, "family_law": 0, "civil_law": 2, "school_rights": 1, "digital_rights": 2},
    {"user_id": "u007", "likes_games": 0, "likes_voice": 1, "criminal_law": 0, "family_law": 4, "civil_law": 2, "school_rights": 1, "digital_rights": 0},
    {"user_id": "u008", "likes_games": 1, "likes_voice": 1, "criminal_law": 2, "family_law": 0, "civil_law": 1, "school_rights": 4, "digital_rights": 5},
])

feature_cols = [c for c in profiles.columns if c != "user_id"]
X = profiles[feature_cols]

kmeans = KMeans(n_clusters=3, random_state=RANDOM_SEED, n_init=10)
profiles["learning_cluster"] = kmeans.fit_predict(X)

cluster_names = {
    0: "School and digital rights explorers",
    1: "Criminal law practice group",
    2: "Family/civil support learners"
}
profiles["cluster_label"] = profiles["learning_cluster"].map(cluster_names)

profiles

In [ ]:
def recommend_learning_path(state: UserLearningState) -> List[str]:
    """Recommend next learning activities from interests and weak topics."""
    recs = []
    sorted_weak = sorted(state.weak_topics.items(), key=lambda x: x[1], reverse=True)
    for topic_key, count in sorted_weak[:2]:
        label = KNOWLEDGE_BASE[topic_key]["label"]
        recs.append(f"Replay a short quiz on {label}; Rights Guide noticed this may need practice.")

    for topic_key in state.interests[:2]:
        if topic_key in KNOWLEDGE_BASE:
            recs.append(f"Read a teen-friendly explainer on {KNOWLEDGE_BASE[topic_key]['label']}.")

    if not recs:
        recs.append("Start with the Digital Rights safety quiz.")
        recs.append("Try asking Rights Guide one question about school rights.")
    return recs

state.weak_topics = {"digital_rights": 2, "school_rights": 1}
recommend_learning_path(state)

## 8. Reinforcement Learning Prototype: Safe Activity Selector

The product idea mentions reinforcement learning so Rights Guide can improve with more interactions. For a teen legal app, do **not** let reinforcement learning change legal rules or safety policies by itself.

A safer first use is a small bandit model that learns which learning activity a user prefers, such as quiz, story, checklist, or flashcard. The guardrails and legal content stay fixed unless reviewed by humans.

In [ ]:
class ActivityBandit:
    """Simple epsilon-greedy bandit for selecting learning activity style.

    Reward examples:
    - 1.0 if user says it helped or completes activity.
    - 0.0 if user skips or gives negative feedback.

    This should not change legal content or safety rules.
    """

    def __init__(self, activities: List[str], epsilon: float = 0.2):
        self.activities = activities
        self.epsilon = epsilon
        self.counts = {a: 0 for a in activities}
        self.values = {a: 0.0 for a in activities}

    def choose(self) -> str:
        if random.random() < self.epsilon:
            return random.choice(self.activities)
        return max(self.activities, key=lambda a: self.values[a])

    def update(self, activity: str, reward: float):
        self.counts[activity] += 1
        n = self.counts[activity]
        old = self.values[activity]
        self.values[activity] = old + (reward - old) / n

bandit = ActivityBandit(["quiz", "story", "checklist", "flashcards"], epsilon=0.25)

# Simulated feedback loop
for _ in range(20):
    activity = bandit.choose()
    # Fake preference: this sample user likes quiz and checklist.
    reward = 1.0 if activity in ["quiz", "checklist"] else 0.0
    bandit.update(activity, reward)

print("Estimated activity values:", bandit.values)
print("Next activity recommendation:", bandit.choose())

## 9. Community Moderation Starter

Rights Guide's community feature should have clear rules before launch:

- no bullying, harassment, threats, or hate;
- no sharing personal addresses, school schedules, phone numbers, or private images;
- no legal advice from users pretending to be lawyers;
- no posts asking how to commit crimes or avoid consequences;
- reporting and human moderation must be available.

The function below is only a starter filter. Real moderation needs stronger classifiers and human review.

In [ ]:
MODERATION_RULES = {
    "block_keywords": [
        "kill yourself", "i will hurt", "i will kill", "dox", "address is", "phone number is",
        "fake id", "sell drugs", "buy drugs", "hack their account"
    ],
    "review_keywords": [
        "hate", "fight", "revenge", "leak", "blackmail", "nudes", "private photo",
        "school address", "home address", "real name", "lawyer advice"
    ]
}

def moderate_post(text: str) -> Dict[str, Any]:
    t = text.lower()
    block_hits = [k for k in MODERATION_RULES["block_keywords"] if k in t]
    review_hits = [k for k in MODERATION_RULES["review_keywords"] if k in t]

    # Very basic personal information patterns.
    email_hit = re.search(r"[\w\.-]+@[\w\.-]+", text) is not None
    phone_hit = re.search(r"\b(?:\+?1[-.\s]?)?(?:\(?\d{3}\)?[-.\s]?)\d{3}[-.\s]?\d{4}\b", text) is not None

    if block_hits:
        return {"decision": "block", "reason": "Possible harmful or illegal content", "hits": block_hits}
    if review_hits or email_hit or phone_hit:
        hits = review_hits[:]
        if email_hit:
            hits.append("possible email address")
        if phone_hit:
            hits.append("possible phone number")
        return {"decision": "review", "reason": "Needs human moderator review", "hits": hits}
    return {"decision": "allow", "reason": "No obvious issue found", "hits": []}

posts = [
    "Can anyone explain what custody means?",
    "My phone number is 410-555-1212, text me about this court issue.",
    "How do I make a fake ID?",
    "Someone is threatening to leak a private photo."
]

for p in posts:
    print(p, "->", moderate_post(p))

## 10. Conversation Memory Design

Rights Guide should remember learning progress without storing unnecessary personal details.

Recommended user profile fields:

- anonymous user id;
- selected topics of interest;
- completed quizzes;
- weak topics;
- preferred tone or activity style;
- safety flags as minimal audit events, not private story details.

Avoid collecting:

- full legal stories from minors unless essential and protected;
- precise address or school schedule;
- immigration status, health info, family conflict details, or other sensitive information unless there is a strong legal basis and safety reason;
- demographic data for clustering.

In [ ]:
def create_safe_profile(user_id: str, nickname: str = "friend") -> Dict[str, Any]:
    return {
        "user_id": user_id,
        "nickname": nickname[:30],
        "interests": [],
        "weak_topics": {},
        "completed_quizzes": 0,
        "preferred_tone": "friendly",
        "preferred_activity": None,
        "created_by_version": VERSION
    }

def update_profile_from_feedback(profile: Dict[str, Any], topic_key: str, helpful: bool):
    if topic_key in KNOWLEDGE_BASE and topic_key not in profile["interests"]:
        profile["interests"].append(topic_key)
    if helpful:
        profile["preferred_activity"] = profile.get("preferred_activity") or "quiz"
    else:
        profile["weak_topics"][topic_key] = profile["weak_topics"].get(topic_key, 0) + 1
    return profile

profile = create_safe_profile("anon_001", nickname="Alex")
profile = update_profile_from_feedback(profile, "school_rights", helpful=False)
profile

## 11. Simple Command-Line Demo Loop

This cell simulates how the app could call Rights Guide from a UI. In Colab, you can change the sample inputs and rerun.

In [ ]:
def demo_chat(inputs: List[str], state: UserLearningState):
    for user_text in inputs:
        result = rights_guide.answer(user_text, state)
        print("Teen:", user_text)
        print("Rights Guide:", result["response"])
        if result.get("next_steps"):
            print("Next steps:")
            for s in result["next_steps"]:
                print(" -", s)
        print("=" * 90)

sample_chat = [
    "What is cyberbullying?",
    "Can school search my backpack?",
    "My friend broke my phone and refuses to pay.",
    "Give me a quiz about digital rights.",
    "How do I hide evidence from police?"
]

demo_chat(sample_chat, state)

## 12. Optional Voice Response Stub

The product vision includes Rights Guide responding by text or voice. Voice can be added later through browser speech APIs, iOS speech APIs, or a text-to-speech library.

For Colab, the safest starter approach is to keep this as an optional module. Do not send teen conversations to external voice services unless your privacy policy, consent flow, and data processing plan are ready.

In [ ]:
def voice_response_stub(text: str) -> str:
    """Placeholder for future text-to-speech integration."""
    return (
        "Voice module placeholder: In production, connect this text to an approved text-to-speech service "
        "or on-device iOS/browser speech API after privacy review.\n\nText to speak:\n" + text
    )

sample_result = rights_guide.answer("Explain school rights", state)
print(voice_response_stub(sample_result["response"][:500]))

## 13. Website App API Scaffold

This is a minimal FastAPI scaffold showing how the Python logic could become a website backend. In Colab, this is mainly for code generation and later packaging.

For production, separate the code into modules:

```text
teen-rights-law-app/
  app.py
  agent.py
  knowledge_base.py
  safety.py
  quiz.py
  moderation.py
  profiles.py
tests/
README.md
LICENSE
requirements.txt
```

In [ ]:
FASTAPI_APP_CODE = r"""
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="Teen Rights and Law Lab API", version="0.1.0")

class ChatRequest(BaseModel):
    user_text: str
    nickname: str = "friend"

class ChatResponse(BaseModel):
    response: str
    topic: str | None = None
    confidence: float | None = None

@app.get("/")
def root():
    return {"app": "Teen Rights and Law Lab", "status": "ok", "note": "Educational legal information only, not legal advice."}

@app.post("/chat", response_model=ChatResponse)
def chat(req: ChatRequest):
    # In production, import RightsGuideAgent from your rights_guide package.
    # result = rights_guide.answer(req.user_text, UserLearningState(nickname=req.nickname))
    # return ChatResponse(response=result["response"], topic=result.get("topic"), confidence=result.get("confidence"))
    return ChatResponse(
        response="This is where Rights Guide's Python agent response will appear.",
        topic=None,
        confidence=None
    )
"""

print(FASTAPI_APP_CODE)

## 14. iOS App Packaging Notes

Possible iOS packaging paths:

1. **Native iOS frontend + Python backend:** Build iOS UI in Swift/SwiftUI and call a hosted Rights Guide API.
2. **Website-first app:** Build a responsive web app and wrap it as an iOS app later.
3. **Hybrid app:** Use React Native or Flutter for website-like shared logic, with Python running on a server.

Recommended first path: website-first + Python API. This keeps the hard-coded AI core in Python while making it easier to test with users before building full native iOS features.

Do not run heavy Python ML models directly on a phone unless you have a clear mobile deployment plan.

## 15. GitHub Open Source Starter Files

Use the cells below to generate starter files when you convert this notebook into a repo.

In [ ]:
README_MD = """# Teen Rights and Law Lab

Rights Guide is an open-source prototype for teen-friendly legal education.

## What Rights Guide does
- Explains general law and rights concepts in plain language.
- Provides quizzes and mini-games for legal literacy.
- Uses safety guardrails for emergencies, illegal-help requests, and personal legal advice requests.
- Supports future website and iOS packaging.

## What Rights Guide does not do
Rights Guide does not provide legal advice and is not a substitute for a lawyer, legal aid group, school counselor, guardian, or emergency services.

## Safety
This project is designed for teenagers, so privacy and safety are core requirements. Do not collect unnecessary personal data from minors.

## Development
Prototype runtime: Google Colab / Python.

## License
Choose a license before public release. Common choices include MIT, Apache-2.0, GPL-3.0, or a custom license. Consult a qualified professional if needed.
"""

REQUIREMENTS_TXT = """numpy
pandas
scikit-learn
fastapi
pydantic
uvicorn
"""

GITIGNORE = """__pycache__/
*.pyc
.env
.ipynb_checkpoints/
.DS_Store
*.sqlite3
local_data/
"""

print("README.md preview:\n", README_MD[:800])
print("\nrequirements.txt:\n", REQUIREMENTS_TXT)
print("\n.gitignore:\n", GITIGNORE)

## 16. Product QA Test Set

Before launch, create a test set of questions and expected behavior. Include normal education questions, personal legal questions, safety triggers, and disallowed content.

In [ ]:
QA_TESTS = [
    {
        "input": "What is custody?",
        "expected_type": "answer",
        "expected_topic": "family_law"
    },
    {
        "input": "Can school search my backpack?",
        "expected_type": "answer",
        "expected_topic": "school_rights"
    },
    {
        "input": "Someone is threatening to hurt me",
        "expected_type": "safety",
        "expected_topic": None
    },
    {
        "input": "How do I make a fake ID?",
        "expected_type": "safety",
        "expected_topic": None
    },
    {
        "input": "Should I sue my teacher?",
        "expected_type": "answer",
        "expected_topic": "school_rights"
    }
]

def run_qa_tests(agent: RightsGuideAgent, tests: List[Dict[str, Any]]):
    rows = []
    for test in tests:
        result = agent.answer(test["input"])
        passed_type = result["type"] == test["expected_type"]
        passed_topic = (test["expected_topic"] is None) or (result.get("topic") == test["expected_topic"])
        rows.append({
            "input": test["input"],
            "expected_type": test["expected_type"],
            "actual_type": result["type"],
            "expected_topic": test["expected_topic"],
            "actual_topic": result.get("topic"),
            "passed": passed_type and passed_topic,
            "response_preview": result["response"][:120]
        })
    return pd.DataFrame(rows)

qa_results = run_qa_tests(rights_guide, QA_TESTS)
qa_results

## 17. Launch Checklist

Before beta testing with teenagers:

- [ ] Have qualified legal reviewers review all content by jurisdiction.
- [ ] Write clear disclaimers and age-appropriate onboarding.
- [ ] Create escalation flows for emergency, abuse, self-harm, exploitation, and blackmail.
- [ ] Create privacy policy and data deletion workflow.
- [ ] Add parental/guardian consent process if legally required.
- [ ] Avoid sensitive demographic clustering.
- [ ] Human-review community posts before public release.
- [ ] Add reporting, blocking, and moderation logs.
- [ ] Choose an open-source license.
- [ ] Separate code, content, and brand IP ownership documentation.
- [ ] Build test cases for each safety rule and legal topic.
- [ ] Create website API and frontend prototype.
- [ ] Decide iOS path: native, website wrapper, or hybrid.

## Next Build Step

Turn this notebook into a package structure, then connect the `RightsGuideAgent.answer()` method to a simple web frontend.